# 00 — Lakehouse Config

## Where we are in the story

**Contoso Retail** runs hundreds of stores. Every night, point-of-sale systems export CSV files — stores, products, and sales lines. In this workshop we land that data in **OneLake**, refine it through a **medallion** pattern (bronze → silver → gold), and eventually serve it to SQL analysts, Power BI executives, and real-time operations teams — **without copying the data**.

This notebook is the **foundation**. Run it once before anything else. Every other notebook in this module starts with `%run 00_config` so they all share the same lakehouse name and schema paths.

---

## What this notebook does

| Step | Purpose |
| --- | --- |
| **Config** | Central place for lakehouse + schema names (mirrors repo `.env`) |
| **Schemas** | Creates `bronze`, `silver`, `gold` namespaces in the lakehouse |
| **Sanity check** | Confirms the default lakehouse is attached and raw CSVs exist |

---

## Before you run anything

1. Open this notebook in workspace **`Fabric-Demo-Workshop`**.
2. In the **Explorer** pane (left), click **Add data items → Existing data sources** (or **Lakehouses +**) and attach **`lh_retail`** as the **default lakehouse**. It must show as **pinned**.
3. Ensure sample CSVs are under `Files/bronze/` — run `pwsh module-0-setup/setup.ps1 -Action data` locally, or upload from `module-0-setup/data/`.
4. Edit the **CONFIG** cell below if your item names differ from `.env`.

> **Presenter note:** Say *"OneLake is OneDrive for data — one storage account for the whole tenant. The lakehouse is our Spark/Python workspace on top of it."* Then attach the lakehouse live on stage.

## Step 1 — Configuration

These variables are the **single source of truth** for every notebook in Module 1. If you rename items in Fabric, update here and in `.env`.

- **`LAKEHOUSE_NAME`** — must match the attached default lakehouse.
- **`BRONZE/SILVER/GOLD_SCHEMA`** — medallion layer names (Unity Catalog-style schemas).
- **`BRONZE_FILES`** — path to raw CSV uploads inside the lakehouse **Files** area (not yet Delta tables).

In [ ]:
# === CONFIG (edit to match your .env) ===
LAKEHOUSE_NAME = "lh_retail"      # must be attached as the DEFAULT lakehouse
BRONZE_SCHEMA  = "bronze"
SILVER_SCHEMA  = "silver"
GOLD_SCHEMA    = "gold"
BRONZE_FILES   = "Files/bronze"   # where the raw CSVs were uploaded
print(f"Config loaded. Default lakehouse should be: {LAKEHOUSE_NAME}")

## Step 2 — Create medallion schemas

Fabric lakehouses support **schemas** (like databases) so we can organize tables into logical layers:

| Schema | Role | Analogy |
| --- | --- | --- |
| **bronze** | Raw landing — preserve source fidelity | The loading dock |
| **silver** | Cleaned, conformed, joined | The quality-control line |
| **gold** | Business-ready aggregates for BI | The showroom |

`CREATE SCHEMA IF NOT EXISTS` is idempotent — safe to re-run.

> **Watch the Explorer:** After this cell, refresh the lakehouse **Tables** view — you should see empty `bronze`, `silver`, `gold` folders appear.

In [ ]:
# Create the medallion schemas (safe to re-run). Requires lakehouse schemas enabled at creation.
for s in [BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {s}")
print("schemas ready:", spark.sql("SHOW SCHEMAS").toPandas()["namespace"].tolist())

## Step 3 — Sanity check (raw files + lakehouse attachment)

Before ingesting, confirm:
1. A **default lakehouse** is attached (otherwise `notebookutils.fs` paths fail).
2. Three CSV files exist: `stores.csv`, `products.csv`, `sales.csv`.

If this fails, attach `lh_retail` and upload data — see `module-0-setup/README.md`.

---

## ✅ Success looks like

- Output lists `bronze`, `silver`, `gold` in `SHOW SCHEMAS`.
- Three CSV files listed under `Files/bronze/`.

**Next notebook:** `01_bronze_ingest` — land the CSVs as Delta tables in bronze.

In [ ]:
# Sanity check: confirm default lakehouse attached and list bronze files.
try:
    files = notebookutils.fs.ls(BRONZE_FILES)
    print(f"Found {len(files)} file(s) under {BRONZE_FILES}:")
    for f in files:
        print(" -", f.name)
except Exception as e:
    print("No default lakehouse attached or no files yet.")
    print("Fix: attach lh_retail, then run setup.ps1 -Action data or upload CSVs manually.")
    print(e)